# 72 Veeva Link spot-check (Phase 1, P5)

Task P5 of `70_veeva_link_phase1_working_plan.md`: compare 10 known HCPs from `31_veeva_kol_list.csv` with their Veeva Link web profiles.

1. **Pick 10 HCPs**: one per profile type (rules in cell 3), all in the HCP Universe, chosen in a fixed order so a re-run picks the same people.
2. **Profile card per HCP**: identity and Veeva IDs, coverage and segment, the `71_` feature values (cohort `2026-10_offset7`, same as every NBA 3.1 cohort from 2025-02), and the underlying rows: trials, latest publications, latest events, and payments by year.
3. **Automatic cross-check**: the counts in `31_veeva_kol_list.csv` (an earlier extract) against the snapshot counts.
4. **Manual step**: open each HCP in Veeva Link (search by name or `LINK_ID`), compare with the card, and fill in `review_template` (`web_matches`, `mismatch_note`). Then copy the mismatches into section 8 of `70_`.

Output: `OUT/features_phase1/spot_check/` (`selection`, `cards`, `review_template`).

In [0]:
from pyspark.sql import functions as F, Window as W
import pandas as pd

OUT = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume/hcp_level_model/dev_pipeline/notebook_output/veeva_link_to_people/"
SNAP, ACT, P1 = f"{OUT}snapshot/load=2025-06-30/", f"{OUT}activity/", f"{OUT}features_phase1/"
SC = P1 + "spot_check/"
KOL_CSV = "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_1_develop/NBA 3.1 - Dev/Veeva Link To People Analysis/31_veeva_kol_list.csv"
COHORT = "2026-10_offset7"
N_ROWS = 5   # rows shown per activity type on a card

kol = pd.read_csv(KOL_CSV, dtype={"SE_ID": str, "NPI": str})
feat = spark.read.parquet(P1 + "t1_t2").filter(F.col("COHORT_ID") == COHORT)
spine = spark.read.parquet(OUT + "spine/veeva_npi_se_map").select("NPI", "SE_ID", "BEST_DECILE", "CP_SEGMENT", "CP_SEG")
se = spark.read.parquet(SNAP + "scientific_expert").select("SE_ID", "LINK_ID", "VEEVA_ID", "FIRST_NAME", "LAST_NAME", "SPECIALTY_1", "DEGREE_1", "EXPERT_TYPE", "JOB_TITLE")

k = spark.createDataFrame(kol[["SE_ID", "NPI", "N_PUBLICATIONS", "N_TRIALS", "N_EVENTS"]]).withColumnRenamed("NPI", "NPI_NUMBER")
pool = (k.join(feat, "NPI_NUMBER").join(spine.withColumnRenamed("NPI", "NPI_NUMBER").drop("SE_ID"), "NPI_NUMBER", "left")
         .filter("veeva_has_profile = 1").toPandas())
print("KOL list rows:", len(kol), "| in HCP Universe and matched:", len(pool), "| SE_ID agrees with spine:",
      int(spark.createDataFrame(pool[["NPI_NUMBER", "SE_ID"]]).join(spine.withColumnRenamed("NPI", "NPI_NUMBER"), ["NPI_NUMBER", "SE_ID"]).count()))

In [0]:
# Each rule is tried in order; ties broken by NPI so the pick is repeatable; an HCP is used once
RULES = [
    ("MRA / finerenone trial investigator", lambda d: (d.veeva_trial_mra_flag_5y == 1) & (d.veeva_trial_pi_flag_5y == 1)),
    ("SGLT2 trial investigator",            lambda d: (d.veeva_trial_sglt2_flag_5y == 1)),
    ("Podium speaker, in-scope events",     lambda d: (d.veeva_event_podium_flag_3y == 1) & (d.veeva_event_n_inscope_3y >= 2)),
    ("Senior author, in-scope publications", lambda d: (d.veeva_pub_senior_author_flag_5y == 1) & (d.veeva_pub_n_inscope_5y >= 5)),
    ("Segment A, decile 10",                lambda d: (d.CP_SEG == "A") & (d.BEST_DECILE == 10)),
    ("Non-target, decile 9 to 10",          lambda d: (d.CP_SEG == "Non-target") & (d.BEST_DECILE >= 9)),
    ("Scored, no trials / pubs / events",   lambda d: (d.veeva_coverage_status == "SCORED") & (d.veeva_trial_any_5y == 0) & (d.veeva_pub_any_5y == 0) & (d.veeva_event_any_3y == 0)),
    ("Paid by a competitor, speaker / consult", lambda d: (d.veeva_pay_competitor_flag_3y == 1) & (d.veeva_pay_speaker_consult_flag_3y == 1)),
    ("Kidney society role",                 lambda d: d.veeva_kidney_society_flag == 1),
    ("Unscored, in Veeva",                  lambda d: d.veeva_coverage_status == "IN_VEEVA_UNSCORED"),
]
picked, used = [], set()
for name, rule in RULES:
    c = pool[rule(pool) & ~pool.NPI_NUMBER.isin(used)].sort_values("NPI_NUMBER")
    if len(c):
        r = c.iloc[0]; used.add(r.NPI_NUMBER)
        picked.append({"profile_type": name, "NPI_NUMBER": r.NPI_NUMBER, "SE_ID": r.SE_ID, "n_candidates": len(c)})
    else:
        picked.append({"profile_type": name, "NPI_NUMBER": None, "SE_ID": None, "n_candidates": 0})
sel = pd.DataFrame(picked)
sel_ok = sel.dropna(subset=["NPI_NUMBER"]).reset_index(drop=True)
sel_ok.insert(0, "card", range(1, len(sel_ok) + 1))
print("picked:", len(sel_ok), "of", len(RULES))
display(sel)

In [0]:
ids = spark.createDataFrame(sel_ok[["card", "profile_type", "NPI_NUMBER", "SE_ID"]])
se_ids = ids.select("SE_ID")

head = (ids.join(se, "SE_ID", "left").join(spine.withColumnRenamed("NPI", "NPI_NUMBER").drop("SE_ID"), "NPI_NUMBER", "left")
           .join(feat.drop("PROVIDER_ID"), "NPI_NUMBER", "left").orderBy("card").toPandas())

tags = ["TAG_FINERENONE", "TAG_MRA", "TAG_SGLT2", "TAG_CKD", "TAG_HF", "TAG_T2D", "TAG_T1D", "TAG_CARDIORENAL"]
def tag_list(df):
    return F.concat_ws(",", *[F.when(F.col(t) == 1, F.lit(t[4:].lower())) for t in tags])

def top(df, order, n=N_ROWS):
    w = W.partitionBy("SE_ID").orderBy(*order)
    return df.join(se_ids, "SE_ID").withColumn("rk", F.row_number().over(w)).filter(f"rk <= {n}").drop("rk")

tr = (top(spark.read.parquet(ACT + "trials"), [F.desc("TAG_ANY_INSCOPE"), F.desc("START_DATE")])
        .select("SE_ID", "NAME", "ROLE", "PHASE", "START_DATE", "END_DATE", tag_list(None).alias("tags")).toPandas())
pb = (top(spark.read.parquet(ACT + "publications"), [F.desc("DATE")])
        .select("SE_ID", "DATE", "AUTHORSHIP_POSITION", "JOURNAL", "TITLE", tag_list(None).alias("tags")).toPandas())
ev = (top(spark.read.parquet(ACT + "events"), [F.desc("START_DATE")])
        .select("SE_ID", "START_DATE", "POSITION", "EVENT_NAME", "TALK_TITLE", tag_list(None).alias("tags")).toPandas())
pay_all = spark.read.parquet(ACT + "payments").join(se_ids, "SE_ID").withColumn("AMOUNT", F.expr("try_cast(AMOUNT AS DOUBLE)"))
py = (pay_all.groupBy("SE_ID", "PROGRAM_YEAR").agg(F.count("*").alias("n_records"), F.round(F.sum("AMOUNT"), 0).alias("usd"),
                                                  F.countDistinct("COMPANY_NAME").alias("n_companies"), F.max("BAYER_FLAG").alias("bayer"))
        .orderBy("SE_ID", F.desc("PROGRAM_YEAR")).toPandas())
pc = (pay_all.groupBy("SE_ID", "COMPANY_NAME").agg(F.round(F.sum("AMOUNT"), 0).alias("usd"))
        .withColumn("rk", F.row_number().over(W.partitionBy("SE_ID").orderBy(F.desc("usd")))).filter("rk <= 3").drop("rk").toPandas())

# all-time snapshot counts for the cross-check with 31_veeva_kol_list.csv
cnt = (ids.select("SE_ID")
       .join(spark.read.parquet(ACT + "publications").groupBy("SE_ID").agg(F.count("*").alias("snap_pubs")), "SE_ID", "left")
       .join(spark.read.parquet(ACT + "trials").groupBy("SE_ID").agg(F.countDistinct("CLINICAL_TRIAL_ID").alias("snap_trials")), "SE_ID", "left")
       .join(spark.read.parquet(ACT + "events").groupBy("SE_ID").agg(F.countDistinct("MEDICAL_EVENT_ID").alias("snap_events")), "SE_ID", "left")
       .fillna(0).toPandas())
print("cards:", len(head), "| trial rows:", len(tr), "| publication rows:", len(pb), "| event rows:", len(ev), "| payment years:", len(py))

In [0]:
FEATS = ["veeva_coverage_status", "veeva_n_inscope_themes", "veeva_trial_any_5y", "veeva_trial_pi_flag_5y", "veeva_trial_n_inscope_5y",
         "veeva_trial_max_phase_inscope_5y", "veeva_trial_mra_flag_5y", "veeva_trial_sglt2_flag_5y", "veeva_pub_any_5y",
         "veeva_pub_senior_author_flag_5y", "veeva_pub_n_inscope_5y", "veeva_pub_months_since_last", "veeva_event_podium_flag_3y",
         "veeva_event_n_inscope_3y", "veeva_pay_any_3y", "veeva_pay_n_companies_3y", "veeva_pay_speaker_consult_flag_3y",
         "veeva_pay_competitor_flag_3y", "veeva_pay_bayer_flag_3y", "veeva_is_academic", "veeva_n_society_roles", "veeva_kidney_society_flag"]
for _, h in head.iterrows():
    s = h.SE_ID
    print("=" * 110)
    print(f"CARD {h.card}: {h.profile_type}")
    print(f"  {h.FIRST_NAME} {h.LAST_NAME} | NPI {h.NPI_NUMBER} | SE_ID {s} | LINK_ID {h.LINK_ID} | VEEVA_ID {h.VEEVA_ID}")
    print(f"  {h.EXPERT_TYPE} | {h.DEGREE_1} | {h.SPECIALTY_1} | {h.JOB_TITLE}")
    print(f"  segment {h.CP_SEGMENT} | best in-scope decile {h.BEST_DECILE} (T3, for review only)")
    print("  features (cohort " + COHORT + "): " + "; ".join(f"{f.replace('veeva_', '')}={h[f]}" for f in FEATS))
    for title, df in [("Trials (in-scope first)", tr), ("Latest publications", pb), ("Latest events", ev), ("Payments by program year", py), ("Top 3 paying companies", pc)]:
        d = df[df.SE_ID == s].drop(columns="SE_ID")
        print(f"  -- {title}: {len(d)} rows shown")
        if len(d):
            print(d.to_string(index=False, max_colwidth=70).replace("\n", "\n     ").join(["     ", ""]))

In [0]:
# 31_ has one row per SE_ID x source list (focus area / clinical leader / digital reach), so keep one row per SE_ID
kol1 = kol.drop_duplicates("SE_ID")[["SE_ID", "N_PUBLICATIONS", "N_TRIALS", "N_EVENTS"]]
x = (sel_ok.merge(kol1, on="SE_ID", how="left").merge(cnt, on="SE_ID", how="left"))
for a, b in [("N_PUBLICATIONS", "snap_pubs"), ("N_TRIALS", "snap_trials"), ("N_EVENTS", "snap_events")]:
    x[f"diff_{b[5:]}"] = x[b] - x[a]
print("Counts in 31_ come from an earlier extract; differences are expected to be small, not zero")
display(x[["card", "profile_type", "NPI_NUMBER", "N_PUBLICATIONS", "snap_pubs", "N_TRIALS", "snap_trials", "N_EVENTS", "snap_events",
           "diff_pubs", "diff_trials", "diff_events"]])

spark.createDataFrame(sel).write.mode("overwrite").parquet(SC + "selection")
cards = head[["card", "profile_type", "NPI_NUMBER", "SE_ID", "LINK_ID", "VEEVA_ID", "FIRST_NAME", "LAST_NAME", "SPECIALTY_1",
              "CP_SEGMENT", "BEST_DECILE"] + FEATS].merge(x[["SE_ID", "snap_pubs", "snap_trials", "snap_events"]], on="SE_ID")
spark.createDataFrame(cards.astype(str)).write.mode("overwrite").parquet(SC + "cards")
template = cards[["card", "profile_type", "NPI_NUMBER", "LINK_ID", "FIRST_NAME", "LAST_NAME"]].assign(
    web_matches="", mismatch_note="", reviewer="", review_date="")
spark.createDataFrame(template.astype(str)).write.mode("overwrite").parquet(SC + "review_template")
assert len(x) == len(sel_ok) == len(cards), (len(x), len(sel_ok), len(cards))
print("written:", SC, "| cards:", len(cards), "| SE_IDs repeated in 31_:", int(kol.SE_ID.duplicated().sum()))
display(template)